In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


Dataset loaded

In [7]:
df = pd.read_csv('Palo Alto Networks.csv')

print(f"Dataset loaded!")
print(f"Shape : {df.shape}")
print()
df.head()

Dataset loaded!
Shape : (1470, 31)



,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


 Check Column Types

In [8]:
# Separate text columns and number columns
text_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols  = df.select_dtypes(include=['number']).columns.tolist()

print("TEXT columns (need to convert to numbers):")
for col in text_cols:
    print(f"  {col:<25} values: {df[col].unique().tolist()}")

print()
print("NUMBER columns (already ready):")
for col in num_cols:
    print(f"  {col}")

TEXT columns (need to convert to numbers):
  BusinessTravel            values: ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
  Department                values: ['Sales', 'Research & Development', 'Human Resources']
  EducationField            values: ['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
  Gender                    values: ['Female', 'Male']
  JobRole                   values: ['Sales Executive', 'Research Scientist', 'Laboratory Technician', 'Manufacturing Director', 'Healthcare Representative', 'Manager', 'Sales Representative', 'Research Director', 'Human Resources']
  MaritalStatus             values: ['Single', 'Married', 'Divorced']
  OverTime                  values: ['Yes', 'No']

NUMBER columns (already ready):
  Age
  Attrition
  DailyRate
  DistanceFromHome
  Education
  EnvironmentSatisfaction
  HourlyRate
  JobInvolvement
  JobLevel
  JobSatisfaction
  MonthlyIncome
  MonthlyRate
  NumCompaniesWorked
  PercentSala

 Encode Text Columns

In [9]:
# Make a copy so original data stays safe
df_clean = df.copy()

# Step 1 — Simple Yes/No and Male/Female columns
# These only have 2 values so we use LabelEncoder
le = LabelEncoder()

df_clean['Gender']   = le.fit_transform(df_clean['Gender'])    # Female=0, Male=1
df_clean['OverTime'] = le.fit_transform(df_clean['OverTime'])  # No=0, Yes=1

print("Gender and OverTime encoded!")
print(f"  Gender  unique values : {df_clean['Gender'].unique()}")
print(f"  OverTime unique values: {df_clean['OverTime'].unique()}")

Gender and OverTime encoded!
  Gender  unique values : [0 1]
  OverTime unique values: [1 0]


Encode Remaining Text Columns

In [10]:
# Step 2 — Columns with more than 2 values
# We use One-Hot Encoding for these
# This creates separate columns for each category

df_clean = pd.get_dummies(df_clean, columns=[
    'BusinessTravel',
    'Department',
    'EducationField',
    'JobRole',
    'MaritalStatus'
])

print("All text columns encoded!")
print(f"New shape : {df_clean.shape}")
print()
print("New columns created:")
new_cols = [col for col in df_clean.columns if col not in df.columns]
for col in new_cols:
    print(f"  {col}")

All text columns encoded!
New shape : (1470, 50)

New columns created:
  BusinessTravel_Non-Travel
  BusinessTravel_Travel_Frequently
  BusinessTravel_Travel_Rarely
  Department_Human Resources
  Department_Research & Development
  Department_Sales
  EducationField_Human Resources
  EducationField_Life Sciences
  EducationField_Marketing
  EducationField_Medical
  EducationField_Other
  EducationField_Technical Degree
  JobRole_Healthcare Representative
  JobRole_Human Resources
  JobRole_Laboratory Technician
  JobRole_Manager
  JobRole_Manufacturing Director
  JobRole_Research Director
  JobRole_Research Scientist
  JobRole_Sales Executive
  JobRole_Sales Representative
  MaritalStatus_Divorced
  MaritalStatus_Married
  MaritalStatus_Single


Separate Features and Target

In [11]:
# X = all columns except Attrition (these are inputs to the model)
# y = only Attrition column (this is what we want to predict)

X = df_clean.drop('Attrition', axis=1)
y = df_clean['Attrition']

print(f"X shape (features) : {X.shape}")
print(f"y shape (target)   : {y.shape}")
print()
print(f"Attrition distribution:")
print(f"  Stayed (0) : {(y==0).sum()} employees")
print(f"  Left   (1) : {(y==1).sum()} employees")

X shape (features) : (1470, 49)
y shape (target)   : (1470,)

Attrition distribution:
  Stayed (0) : 1233 employees
  Left   (1) : 237 employees


 Split Data into Train and Test

In [12]:
# Split data — 80% for training, 20% for testing
# train = model learns from this
# test  = model is evaluated on this (model never sees this during training)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # ensures same 84:16 ratio in both train and test
)

print("Data split done!")
print()
print(f"Training set   : {X_train.shape[0]} employees  (80%)")
print(f"Testing set    : {X_test.shape[0]} employees  (20%)")
print()
print(f"Training Attrition:")
print(f"  Stayed (0) : {(y_train==0).sum()}")
print(f"  Left   (1) : {(y_train==1).sum()}")

Data split done!

Training set   : 1176 employees  (80%)
Testing set    : 294 employees  (20%)

Training Attrition:
  Stayed (0) : 986
  Left   (1) : 190


Apply SMOTE

In [13]:
# SMOTE creates fake but realistic examples of minority class (employees who left)
# So the model gets equal examples of both stayed and left

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("SMOTE applied successfully!")
print()
print("BEFORE SMOTE (training set):")
print(f"  Stayed (0) : 986")
print(f"  Left   (1) : 190")
print()
print("AFTER SMOTE (training set):")
print(f"  Stayed (0) : {(y_train_sm==0).sum()}")
print(f"  Left   (1) : {(y_train_sm==1).sum()}")
print()
print(f"New training size : {len(X_train_sm)} employees")

SMOTE applied successfully!

BEFORE SMOTE (training set):
  Stayed (0) : 986
  Left   (1) : 190

AFTER SMOTE (training set):
  Stayed (0) : 986
  Left   (1) : 986

New training size : 1972 employees


Scale the Numbers

In [14]:
# StandardScaler brings all numbers to same range
# Example: Age (18-60) and Income (1000-20000) become comparable
# This helps ML models perform better

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done!")
print()
print(f"Training set shape : {X_train_scaled.shape}")
print(f"Testing set shape  : {X_test_scaled.shape}")
print()
print("Sample values BEFORE scaling:")
print(f"  Age range    : {X_train_sm['Age'].min()} to {X_train_sm['Age'].max()}")
print(f"  Income range : {X_train_sm['MonthlyIncome'].min()} to {X_train_sm['MonthlyIncome'].max()}")
print()
print("After scaling all values are between -3 and +3")
print("Preprocessing is 100% COMPLETE!")

Scaling done!

Training set shape : (1972, 49)
Testing set shape  : (294, 49)

Sample values BEFORE scaling:
  Age range    : 18 to 60
  Income range : 1009 to 19973

After scaling all values are between -3 and +3
Preprocessing is 100% COMPLETE!


Final Summary

In [16]:
print("=" * 55)
print("  PREPROCESSING SUMMARY — PHASE 3 COMPLETE")
print("=" * 55)
print()
print("  STEP 1 — Encoding")
print("  Gender, OverTime    : Label Encoded (0 and 1)")
print("  BusinessTravel      : One-Hot Encoded (3 columns)")
print("  Department          : One-Hot Encoded (3 columns)")
print("  EducationField      : One-Hot Encoded (6 columns)")
print("  JobRole             : One-Hot Encoded (9 columns)")
print("  MaritalStatus       : One-Hot Encoded (3 columns)")
print()
print("  STEP 2 — Train Test Split")
print("  Training set : 1176 employees (80%)")
print("  Testing set  :  294 employees (20%)")
print()
print("  STEP 3 — SMOTE")
print("  Before : Stayed 986  Left 190  (imbalanced)")
print("  After  : Stayed 986  Left 986  (balanced!)")
print()
print("  STEP 4 — Scaling")
print("  All features scaled to same range")
print()
print("=" * 55)
print("  NEXT PHASE 4 — BUILD ML MODELS!")
print("=" * 55)

  PREPROCESSING SUMMARY — PHASE 3 COMPLETE

  STEP 1 — Encoding
  Gender, OverTime    : Label Encoded (0 and 1)
  BusinessTravel      : One-Hot Encoded (3 columns)
  Department          : One-Hot Encoded (3 columns)
  EducationField      : One-Hot Encoded (6 columns)
  JobRole             : One-Hot Encoded (9 columns)
  MaritalStatus       : One-Hot Encoded (3 columns)

  STEP 2 — Train Test Split
  Training set : 1176 employees (80%)
  Testing set  :  294 employees (20%)

  STEP 3 — SMOTE
  Before : Stayed 986  Left 190  (imbalanced)
  After  : Stayed 986  Left 986  (balanced!)

  STEP 4 — Scaling
  All features scaled to same range

  NEXT PHASE 4 — BUILD ML MODELS!
